In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("m1_trajectory.ipynb")

# M1 — Trajectory methods and the first swarm

**TC6035 Part 1 · Milestone 1 · due after Session 2**

Memory becomes explicit, then plural. You will implement tabu search over the
discrete layer, PSO over the continuous layer on the subset tabu produced, and
compare tabu against the annealing you built in M0.

> **This milestone must cite M0.** Two values specifically: your
> `MC_BASELINE_MEDIAN` and your `NOMINAL_POWER`. A milestone that cites nothing
> from its predecessor is not scored.

> **Implement in `src/solutions.py`.** Register everything in `ALGORITHMS` and
> import from there. Algorithms defined only inside this notebook cannot be
> re-executed, and reproducibility is 15 of the 60 automatic points.

In [ ]:
import numpy as np
from scipy.stats import wilcoxon

from tc6035 import build_instance, SensorPlacementProblem
from tc6035.runlog import RunSet, save_runset, load_runset
from solutions import ALGORITHMS

STUDENT_ID = "A01234567"   # <-- yours
SALT = 0
BUDGET, N_SEEDS = 5_000, 30

instance = build_instance(STUDENT_ID, salt=SALT)

# --- carried forward from M0 -------------------------------------------------
# Replace with YOUR M0 values. These are cited by the autograder's chaining check.
MC_BASELINE_MEDIAN = 0.5834    # M0 Question 6
NOMINAL_POWER      = 14.0      # M0 Question 5

print(f"M0 carried forward: baseline {MC_BASELINE_MEDIAN:.4f}, nominal power {NOMINAL_POWER}")

---
## Question 1 — Tabu search over the discrete layer

Implement `tabu_search(problem, seed, **params)` in `src/solutions.py`.

Power is frozen at your `NOMINAL_POWER` from M0; only activation is searched.

You must decide three things and **justify each in Question 2**:

- **tenure** — how many iterations a flipped index stays forbidden
- **aspiration** — when a tabu move is allowed anyway
- **candidate list size** — the full neighbourhood is 80 evaluations per step,
  which buys only 62 steps out of your whole budget

Run over `N_SEEDS` seeds and save the run log.

In [ ]:
tabu_records, tabu_scores = [], []
for seed in range(N_SEEDS):
    prob = SensorPlacementProblem(instance, budget=BUDGET, seed=seed)
    tabu_scores.append(ALGORITHMS["tabu_search"](
        prob, seed, tenure= ...
    tabu_records.append(prob.record)
tabu_scores = np.array(tabu_scores)

save_runset("results/m1_tabu.npz", RunSet(algorithm="tabu_search", student_id=STUDENT_ID,
            params={"tenure": 12, "nominal_power": NOMINAL_POWER}, records=tabu_records))

print(f"tabu median {np.median(tabu_scores):.4f}  "
      f"IQR [{np.percentile(tabu_scores,25):.4f}, {np.percentile(tabu_scores,75):.4f}]")
print(f"vs M0 Monte Carlo baseline {MC_BASELINE_MEDIAN:.4f}: "
      f"{np.median(tabu_scores) - MC_BASELINE_MEDIAN:+.4f}")

In [ ]:
grader.check("q1_tabu")

---
## Question 2 — Justify the tenure, do not assert it

Run tabu at several tenures over at least 10 seeds each and record
`tenure_medians`. Then answer, in the markdown cell below: what fails at a
tenure that is too short, and what fails at one that is too long?

> Budget note: this is tuning. Say in Question 5 how you kept it from
> contaminating your reported comparison.

In [ ]:
TENURES = [1, 4, 12, 30, 60]
tenure_medians = ...
...
    vals = ...
    ...
        prob = ...
        ...
            ...
    tenure_medians[tenure] = ...

for t, m in tenure_medians.items():
    print(f"  tenure {t:>3}: median {m:.4f}")

In [ ]:
grader.check("q2_tenure")

<!-- BEGIN QUESTION -->

Explain your tenure choice from the numbers above. What fails at
tenure 1, and what fails at tenure 60?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 3 — PSO over the continuous layer

Implement `pso(problem, seed, **params)` in `src/solutions.py`, searching
transmit power with the subset held fixed.

Report your $(w, c_1, c_2)$ and swarm size. Draw $r_1, r_2$ **per dimension**,
not per particle — per particle turns the cognitive and social terms into a
straight line toward the best, which is a different algorithm.

Run over `N_SEEDS` seeds and save the log.

In [ ]:
W_INERTIA, C1, C2, SWARM = ...

pso_records, pso_scores = [], []
for seed in range(N_SEEDS):
    prob = SensorPlacementProblem(instance, budget=BUDGET, seed=seed)
    ...
    pso_records.append(prob.record)
pso_scores = np.array(pso_scores)

save_runset("results/m1_pso.npz", RunSet(algorithm="pso", student_id=STUDENT_ID,
            params={"swarm_size": SWARM}, records=pso_records))

phi = C1 + C2
print(f"PSO median {np.median(pso_scores):.4f}")
print(f"(w, c1, c2) = ({W_INERTIA}, {C1}, {C2})   phi = {phi:.3f}")
print(f"stability: w<1 and 0<phi<2(1+w)={2*(1+W_INERTIA):.3f}  ->  "
      f"{W_INERTIA < 1 and 0 < phi < 2*(1+W_INERTIA)}")

In [ ]:
grader.check("q3_pso")

---
## Question 4 — Tabu against your M0 annealing

Both are trajectory methods; they differ in whether they remember. Load your M0
annealing log, compare it against tabu over the paired seeds, and store
`p_tabu_vs_sa`.

One comparison, paired, non-normal outcomes. Choose the test accordingly.

In [ ]:
sa_runs = load_runset("results/m0_annealing.npz")
sa_scores = sa_runs.best_values

W_stat, p_tabu_vs_sa = ...
effect = ...

print(f"tabu median {np.median(tabu_scores):.4f}   SA median {np.median(sa_scores):.4f}")
print(f"median paired difference {effect:+.4f}")
print(f"tabu wins {int((tabu_scores > sa_scores).sum())}/{N_SEEDS}")
print(f"Wilcoxon W={W_stat:.1f}  p={p_tabu_vs_sa:.3e}")

In [ ]:
grader.check("q4_compare")

<!-- BEGIN QUESTION -->

---
## Question 5 — Analysis

300–450 words. Address all four:

1. **What did memory buy?** Compare tabu against your M0 annealing
   *mechanistically*, using the landscape numbers you measured in M0.
2. **Your candidate-list decision.** Full neighbourhood is 80 evaluations per
   step; sampling buys more steps. Justify your choice from your $\rho_1$.
3. **PSO's scope.** It searched power on a fixed subset while annealing searched
   both layers. What does that do to the comparison, and what would you have to
   change to make it fair?
4. **Tuning hygiene.** Question 2 spent budget on tuning. How did you keep that
   from contaminating your reported comparison?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Before you submit

- [ ] `src/solutions.py` exposes `tabu_search` and `pso` in `ALGORITHMS`
- [ ] `results/` holds `m1_tabu.npz` and `m1_pso.npz`
- [ ] This notebook cites **M0** by name — the chaining check looks for it
- [ ] Carry forward to M2: tuned tabu configuration and $(w, c_1, c_2, \text{swarm})$
- [ ] `python scripts/self_check.py` passes